# CatVTON Lightweight Baseline

This notebook evaluates CatVTON on a free-tier GPU (like Colab T4 16GB) to measure VRAM and inference speed.

In [ ]:
# 1. Clone the repository and install dependencies
!git clone https://github.com/Zheng-Chong/CatVTON.git
%cd CatVTON
!pip install -q -r requirements.txt
!pip install -q diffusers accelerate xformers triton

In [ ]:
# 2. Download sample images
import urllib.request
import os
os.makedirs('sample_inputs', exist_ok=True)
urllib.request.urlretrieve('https://raw.githubusercontent.com/Zheng-Chong/CatVTON/main/resource/demo/person/1.jpg', 'sample_inputs/person.jpg')
urllib.request.urlretrieve('https://raw.githubusercontent.com/Zheng-Chong/CatVTON/main/resource/demo/garment/1.jpg', 'sample_inputs/garment.jpg')

In [ ]:
# 3. Run Inference and measure speed/memory
import time
import torch
import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

start_time = time.time()

# Run CatVTON natively (Assuming official setup)
# In the official repo they provide a diffusers pipeline interface or an app.py
!python inference.py \
  --image_path sample_inputs/person.jpg \
  --mask_path sample_inputs/person.jpg \
  --reference_path sample_inputs/garment.jpg \
  --output_dir output \
  --mixed_precision fp16

end_time = time.time()
peak_memory = torch.cuda.max_memory_allocated() / (1024 ** 3)

print("-" * 40)
print(f"⏱️ Inference Time: {end_time - start_time:.2f} seconds")
print(f"💾 Peak VRAM Usage: {peak_memory:.2f} GB")
print("-" * 40)

In [ ]:
# 4. Display the result
from PIL import Image
import matplotlib.pyplot as plt

person = Image.open('sample_inputs/person.jpg')
garment = Image.open('sample_inputs/garment.jpg')
import glob
out_images = glob.glob('output/*.png') + glob.glob('output/*.jpg')
if out_images:
    result = Image.open(out_images[0])
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(person); axes[0].set_title('Person'); axes[0].axis('off')
    axes[1].imshow(garment); axes[1].set_title('Garment'); axes[1].axis('off')
    axes[2].imshow(result); axes[2].set_title('Result'); axes[2].axis('off')
    plt.show()
else:
    print("Output image not found.")